# 05 - Fine-Tuning with PEFT, TRL, and Conditional Unsloth

This notebook explains and runs the training stack used in this project.


## Framework Coverage: PEFT, TRL, Unsloth

### PEFT
- Definition: Adapter-based parameter-efficient fine-tuning (LoRA).
- Why used: local memory-efficient tuning for 3B-scale models.
- Where used: all training backends in this project.

### TRL
- Definition: SFT-oriented training stack (`SFTTrainer`, `SFTConfig`).
- Why used: instruction-tuning ergonomics and explicit completion loss configuration.
- Where used: primary backend when available.

### Unsloth
- Definition: accelerated training runtime integrated with TRL workflows.
- Why used: potential speed/VRAM gains only under verified compatibility.
- Where used: optional backend behind strict compatibility gating; not forced.


## What Is This Technique?

### What is this technique?
Hybrid backend fine-tuning (HF/TRL/Unsloth) with guarded fallback

### Definition and core concepts.
A backend router that selects the best available training path while preserving correctness and reproducibility.

### Why was this technique developed?
Training environments vary; a single rigid backend can fail due dependency/runtime incompatibilities.

### What limitations of traditional RAG does it solve?
RAG-only systems avoid training but remain limited by prompt behavior. This technique adds domain adaptation while retaining local deployability.

### Architecture and workflow diagram explanation.
Resolve backend -> prepare instruction dataset -> attach PEFT adapters -> train -> evaluate -> persist adapters + metadata.

### Component-by-component breakdown.
Backend resolver, PEFT config, TRL/HF trainers, optional Unsloth gate, MLflow logging, artifact writer.

### When should it be used in real-world systems?
Use in production pipelines where reliability matters more than forcing one framework choice.

### Advantages and disadvantages.
Advantages: resilient execution and explicit traceability. Disadvantages: extra orchestration complexity.

### Comparison against standard RAG and other implemented RAG variants.
Compared with single-backend training, this design is more robust across environments. Compared with standard RAG-only setups, it enables schema-specific behavioral adaptation.

### Implementation details and design decisions used in this project.
Granite4.1 remains the primary model. Unsloth is only used when compatibility checks pass; otherwise fallback is logged.


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from repo_query_gen.training import run_finetuning

# Fast profile for reproducible tutorial run.
out = run_finetuning("fast", backend="auto", allow_fallback=True)
out


In [ ]:
import json
from pathlib import Path

run_dir = Path(out["run_dir"])
print(json.loads((run_dir / "training_metadata.json").read_text()))
print(json.loads((run_dir / "train_result.json").read_text()))
print(json.loads((run_dir / "eval_result.json").read_text()))


## Post-Run Analysis (PEFT + TRL + conditional Unsloth)

After executing the real pipeline, use this section to document measured outcomes.

- Analyze actual outputs, metrics, retrieval quality, latency, and generation quality.
- Explain how the technique changed measured behavior vs baseline.
- Interpret every metric in business and engineering terms.
- Capture failure modes, lessons learned, and practical takeaways.
- Explain why specific outputs were produced and what they demonstrate.
- Conclude effectiveness based on measured evidence, not assumptions.
